# MCTS Hyperparameter Tuning

Systematic exploration of MCTS hyperparameters to find the best balance between wall time and solution quality.
Track: `MCTS_EPOCHS`, `max_depth`, `valid_weight_proportion`

In [ ]:
from pathlib import Path
import importlib
import random
import sys
import time
import itertools

import numpy as np
import pandas as pd

# Find the project root (the folder that contains src/cma) by walking upward
# from the current working directory. This works whether the notebook is run
# from notebooks/ or from the repo root. If you have run `pip install -e .`,
# this sys.path setup is harmless and not strictly needed.
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'cma').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import cma.servicegraph as servicegraph_module
importlib.reload(servicegraph_module)

from cma.data_reader import (
    read_vessel_class_data,
    read_port_data,
    read_sailing_distance_data,
    read_demand_with_transit_time,
    read_cnc_proforma_data,
)
from cma.mcts import MonteCarloTree
from cma.port import PortGraph
from cma.servicegraph import ServiceGraph

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

print('Imports successful')

## 1. Load Data & Setup (Reuse from test_mcts_kickstart.ipynb)

In [2]:
vesselpool = read_vessel_class_data()
portpool_main, portpool_dmd = read_port_data()
dist_matrix = read_sailing_distance_data(portpool_main)
demand_matrix, transit_time_matrix = read_demand_with_transit_time(portpool_main)

proforma = read_cnc_proforma_data(portpool_main, vesselpool)
all_service_lines = proforma['lines']

print(f'Loaded {len(all_service_lines)} proforma service lines')
print(f'Loaded {portpool_main.get_number_of_ports()} ports')
print(f'Loaded {int((np.asarray(demand_matrix) > 0).sum())} positive OD demands')

Loaded 31 proforma service lines
Loaded 182 ports
Loaded 741 positive OD demands


## 2. Build Smoke-Test Problem

In [3]:
SMOKE_LINE_COUNT = 8
SMOKE_OD_COUNT = 20

full_portgraph = PortGraph(
    portpool_main,
    dist_matrix,
    demand_matrix,
    mat_transit_time=transit_time_matrix,
    filter_by_demand=False,
)

line_count = min(SMOKE_LINE_COUNT, len(all_service_lines))
while True:
    service_lines = all_service_lines[:line_count]
    servicegraph = ServiceGraph(service_lines)
    _, feasible_actions = servicegraph.get_feasible_actions(full_portgraph)
    paths_dict = servicegraph.get_all_paths(full_portgraph, full_portgraph.filtered_by_transship_capacity())
    connected_positive = [
        (od, demand, paths)
        for od, demand, paths in zip(
            paths_dict['od_pairs'],
            paths_dict['od_pairs_demand'],
            paths_dict['od_pairs_path'],
        )
        if demand > 0 and len(paths) > 0
    ]
    if feasible_actions and connected_positive:
        break
    line_count += 2
    if line_count > len(all_service_lines):
        raise RuntimeError('Could not find a service-line subset with feasible MCTS actions and connected demand.')

demand_smoke = np.zeros_like(np.asarray(demand_matrix, dtype=float))
for (origin_idx, dest_idx), demand, _ in connected_positive[:SMOKE_OD_COUNT]:
    demand_smoke[origin_idx, dest_idx] = demand

portgraph = PortGraph(
    portpool_main,
    dist_matrix,
    demand_smoke,
    mat_transit_time=transit_time_matrix,
    filter_by_demand=False,
)

print(f'Using {len(service_lines)} service lines')
print(f'Using {len(demand_smoke[demand_smoke > 0])} OD pairs with demand')
print(f'Smoke weekly demand: {demand_smoke.sum():,.0f} TEU')

Using 8 service lines
Using 20 OD pairs with demand
Smoke weekly demand: 21,260 TEU


## 3. MILP Settings

In [4]:
USE_ACCURATE_DISCRETE_SPEED = False

WEEK_LEVELS = [1, 2, 3, 4, 5, 6, 7]

MCTS_MILP_TUNEPARAMS = {
    'turnon-transship_shipclass_restriction': 0,
    'turnon-vessel_speed_optimization': 0 if USE_ACCURATE_DISCRETE_SPEED else 1,
    'turnon-tight_line_capacity_linearization': 1,
    'turnon-port_operations_constraint': 1,
    'turnon-transit_time_penalty': 1,
    'turnon-demand_fulfillment_cap': 1,
    'ctrparam-kts_buffer': 0,
    'ctrparam-transship_A': 100,
    'ctrparam-speed_soft_cap_kts': 16.5,
    'ctrparam-speed_penalty_multiplier': 2.0,
    'ctrparam-transit_penalty_multiplier': 1000.0,
    'ctrparam-buffer_penalty_below_15pct': 1000.0,
    'ctrparam-buffer_penalty_above_30pct': 20000.0,
    'unfulfilled_demand_penalty': 1e6,
    'BigM-transship': 10000,
    'BigM-n_ships': 10,
    'BigM-saildays': 100,
    'BigM-line_capacity': 30000,
    'BigM-portcall_cost': 1e7,
    'turnon-schedule_adherence': 0,
    'schedule_buffer_hrs': 120.0,
    'solver-MIPGap': 0.25,
    'solver-TimeLimit': 90,
    'solver-MIPFocus': 1,
    'solver-verbose': False,
}

print('MILP params set')

MILP params set


## 4. Baseline Evaluation

In [5]:
print('Computing baseline...\n')
start_time = time.time()
baseline_solution = servicegraph.solve_approximated(
    portgraph,
    vesselpool,
    min_cost=True,
    week_levels=WEEK_LEVELS,
    tuneparams_2=MCTS_MILP_TUNEPARAMS,
)
baseline_elapsed = time.time() - start_time
baseline_cost = servicegraph.total_cost()

print(f'Baseline solve time: {baseline_elapsed:.2f}s')
print(f'Baseline cost: ${baseline_cost:,.2f}')
print(f'Buffer >30% lines: {baseline_solution.get("kpi_lines_buffer_above_30", 0)}')

Computing baseline...

Baseline solve time: 33.58s
Baseline cost: $10,063,777,685.65
Buffer >30% lines: 0


## 5. Define Hyperparameter Grid

In [6]:
# Define ranges to test (smaller subset for faster iteration)
epochs_range = [6, 15, 20]            # Start at 6 (baseline), try 15 and 20
max_depth_range = [2, 3, 4]           # Start at 2 (baseline), try deeper
valid_weight_range = [0.74, 0.85]     # 0.74 is baseline, try higher

# Generate all combinations
hyperparams = list(itertools.product(epochs_range, max_depth_range, valid_weight_range))

print(f'Total hyperparameter combinations to test: {len(hyperparams)}')
print('Estimated runtime: 1-2 hours\n')
print('Combinations:')
for i, (ep, depth, weight) in enumerate(hyperparams, 1):
    print(f'  {i}. epochs={ep}, max_depth={depth}, valid_weight={weight}')

Total hyperparameter combinations to test: 18
Estimated runtime: 1-2 hours

Combinations:
  1. epochs=6, max_depth=2, valid_weight=0.74
  2. epochs=6, max_depth=2, valid_weight=0.85
  3. epochs=6, max_depth=3, valid_weight=0.74
  4. epochs=6, max_depth=3, valid_weight=0.85
  5. epochs=6, max_depth=4, valid_weight=0.74
  6. epochs=6, max_depth=4, valid_weight=0.85
  7. epochs=15, max_depth=2, valid_weight=0.74
  8. epochs=15, max_depth=2, valid_weight=0.85
  9. epochs=15, max_depth=3, valid_weight=0.74
  10. epochs=15, max_depth=3, valid_weight=0.85
  11. epochs=15, max_depth=4, valid_weight=0.74
  12. epochs=15, max_depth=4, valid_weight=0.85
  13. epochs=20, max_depth=2, valid_weight=0.74
  14. epochs=20, max_depth=2, valid_weight=0.85
  15. epochs=20, max_depth=3, valid_weight=0.74
  16. epochs=20, max_depth=3, valid_weight=0.85
  17. epochs=20, max_depth=4, valid_weight=0.74
  18. epochs=20, max_depth=4, valid_weight=0.85


## 6. Run Tuning Experiments

⏱️ **Estimated time: 1-2 hours** (18 combinations of MCTS with MILP solves)

In [ ]:
results = []

for run_num, (mcts_epochs, max_depth, valid_weight) in enumerate(hyperparams, start=11):
    print(f'\n[{run_num}/{len(hyperparams)}] Testing: epochs={mcts_epochs}, depth={max_depth}, weight={valid_weight:.2f}')
    
    try:
        # Create a fresh graph copy for this run
        fresh_servicegraph = ServiceGraph(service_lines)
        
        # Run MCTS
        tree = MonteCarloTree(
            servicegraph=fresh_servicegraph,
            portgraph=portgraph,
            vesselpool=vesselpool,
            discount_fac=0.5,
            valid_weight_proportion=valid_weight,
            max_depth=max_depth,
            c_param=1e-2,
            min_cost=True,
            week_levels=WEEK_LEVELS,
            milp_tuneparams=MCTS_MILP_TUNEPARAMS,
        )
        
        start_time = time.time()
        tree.run(mcts_epochs, display=False)  # No verbose output for cleaner results
        elapsed = time.time() - start_time
        
        # Get results
        best_node = tree.get_best_node()
        best_cost = best_node.graph.total_cost()
        root_cost = tree.root_node.graph.total_cost()
        improvement = root_cost - best_cost
        improvement_pct = (improvement / root_cost) * 100 if root_cost > 0 else 0
        
        results.append({
            'epochs': mcts_epochs,
            'max_depth': max_depth,
            'valid_weight': valid_weight,
            'wall_time': elapsed,
            'root_cost': root_cost,
            'best_cost': best_cost,
            'improvement_usd': improvement,
            'improvement_pct': improvement_pct,
            'tree_nodes': tree.total_number_of_nodes(),
            'expansions': tree.recorder().get('num_expand', 0),
        })
        
        print(f'  Wall time: {elapsed:.2f}s | Best cost: ${best_cost:,.0f} | Improvement: ${improvement:,.0f} ({improvement_pct:.2f}%)')
        
    except Exception as e:
        print(f'  ERROR: {str(e)}')
        results.append({
            'epochs': mcts_epochs,
            'max_depth': max_depth,
            'valid_weight': valid_weight,
            'wall_time': None,
            'best_cost': None,
            'error': str(e),
        })

print('\n✓ Tuning experiments complete!')


[11/18] Testing: epochs=6, depth=2, weight=0.74
  Wall time: 260.22s | Best cost: $10,678,777,334 | Improvement: $0 (0.00%)

[12/18] Testing: epochs=6, depth=2, weight=0.85


## 7. Analyze Results

In [ ]:
results_df = pd.DataFrame(results)

# Filter out failed runs
results_df = results_df[results_df['wall_time'].notna()].copy()

# Display full results sorted by wall time
print('\n=== FULL RESULTS (sorted by wall time) ===')
display_cols = ['epochs', 'max_depth', 'valid_weight', 'wall_time', 'improvement_usd', 'improvement_pct', 'tree_nodes']
print(results_df[display_cols].sort_values('wall_time').to_string(index=False))

print('\n\n=== TOP 5 BY IMPROVEMENT (for quality) ===')
top_quality = results_df.nlargest(5, 'improvement_usd')[display_cols]
print(top_quality.to_string(index=False))

print('\n\n=== TOP 5 BY SPEED (shortest wall time) ===')
top_speed = results_df.nsmallest(5, 'wall_time')[display_cols]
print(top_speed.to_string(index=False))

print('\n\n=== PARETO FRONTIER (best speed-quality tradeoffs) ===')
# Simple heuristic: sort by efficiency (improvement per second)
results_df['improvement_per_sec'] = results_df['improvement_usd'] / results_df['wall_time']
pareto = results_df.nlargest(5, 'improvement_per_sec')[['epochs', 'max_depth', 'valid_weight', 'wall_time', 'improvement_usd', 'improvement_per_sec']]
print(pareto.to_string(index=False))

## 8. Summary & Recommendation

In [ ]:
if len(results_df) > 0:
    # Best by efficiency
    best_idx = results_df['improvement_per_sec'].idxmax()
    best_config = results_df.loc[best_idx]
    
    print(f"\n🎯 RECOMMENDED HYPERPARAMETERS (best speed-quality balance):")
    print(f"   epochs: {int(best_config['epochs'])}")
    print(f"   max_depth: {int(best_config['max_depth'])}")
    print(f"   valid_weight_proportion: {best_config['valid_weight']:.2f}")
    print(f"\n   Wall time: {best_config['wall_time']:.2f}s")
    print(f"   Cost improvement: ${best_config['improvement_usd']:,.0f} ({best_config['improvement_pct']:.2f}%)")
    print(f"   Tree nodes explored: {int(best_config['tree_nodes'])}")
else:
    print('No successful runs to analyze!')

In [ ]:
# Quick summary statistics
print('='*80)
print('SUMMARY STATISTICS')
print('='*80)

if len(results_df) > 0:
    print(f'\nBaseline (MILP only): ${baseline_cost:,.2f} ({baseline_elapsed:.2f}s)')
    print(f'\nMCTS Results ({len(results_df)} successful runs):')
    print(f'  Fastest run: {results_df["wall_time"].min():.2f}s')
    print(f'  Slowest run: {results_df["wall_time"].max():.2f}s')
    print(f'  Average time: {results_df["wall_time"].mean():.2f}s')
    print(f'\n  Best improvement: ${results_df["improvement_usd"].max():,.0f} ({results_df.loc[results_df["improvement_usd"].idxmax(), "improvement_pct"]:.2f}%)')
    print(f'  Avg improvement: ${results_df["improvement_usd"].mean():,.0f} ({results_df["improvement_pct"].mean():.2f}%)')
    print(f'  Worst improvement: ${results_df["improvement_usd"].min():,.0f} ({results_df.loc[results_df["improvement_usd"].idxmin(), "improvement_pct"]:.2f}%)')
    
    print(f'\n  Tree nodes explored: {int(results_df["tree_nodes"].min())} to {int(results_df["tree_nodes"].max())} (avg: {results_df["tree_nodes"].mean():.0f})')
    
    # Best by efficiency
    best_idx = results_df['improvement_per_sec'].idxmax()
    best_config = results_df.loc[best_idx]
    
    print(f'\n' + '='*80)
    print(f'🎯 RECOMMENDED CONFIG (best speed-quality balance):')
    print(f'='*80)
    print(f'   MCTS_EPOCHS: {int(best_config["epochs"])}')
    print(f'   max_depth: {int(best_config["max_depth"])}')
    print(f'   valid_weight_proportion: {best_config["valid_weight"]:.2f}')
    print(f'\n   Wall time: {best_config["wall_time"]:.2f}s')
    print(f'   Cost improvement: ${best_config["improvement_usd"]:,.0f} ({best_config["improvement_pct"]:.2f}%)')
    print(f'   Efficiency: ${best_config["improvement_per_sec"]:,.0f}/sec')
    print(f'   Tree nodes: {int(best_config["tree_nodes"])}')
    print(f'\n   → Use this for Phase 2 (CSV output script)')
else:
    print('No successful runs to summarize.')

## 10. Quick Summary Stats

In [ ]:
from datetime import datetime

# Create output directory if it doesn't exist
output_dir = PROJECT_ROOT / 'tuning_results'
output_dir.mkdir(exist_ok=True)

# Save with timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = output_dir / f'mcts_tuning_results_{timestamp}.csv'

# Save full results (including failed runs)
results_df_all = pd.DataFrame(results)
results_df_all.to_csv(csv_path, index=False)

print(f'✓ Results saved to: {csv_path}')
print(f'  Total runs: {len(results_df_all)}')
print(f'  Successful: {results_df_all["wall_time"].notna().sum()}')
print(f'  Failed: {results_df_all["wall_time"].isna().sum()}')

## 9. Save Results to CSV